#### Import the needed libraries

In [1]:
import os
import glob
import cv2
import torch
torch.set_float32_matmul_precision('high')
import numpy as np
import pandas as pd
import supervision as sv
import torchvision
from torchvision import transforms
from torchvision.ops import box_convert
from torchvision.ops import nms
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image
from sam2.build_sam import build_sam2_video_predictor, build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor 
from groundingdino.util.inference import load_model, load_image, predict, annotate
# Add the parent directory of 'utils' to the Python path
import sys
# Go 3 levels up to reach 'utils'
three_levels_up = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.append(three_levels_up)
from utils.track_utils import sample_points_from_masks
from utils.video_utils import create_video_from_images
import matplotlib.pyplot as plt
from transformers import (
    PaliGemmaProcessor,
    PaliGemmaForConditionalGeneration,
    Gemma3ForConditionalGeneration,
    AutoProcessor
)
#from transformers.image_utils import load_image

from ensemble_boxes import weighted_boxes_fusion
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "C:/torch_temp"

from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from sklearn.metrics.pairwise import cosine_similarity
import ast
import gc
from torchmetrics.detection.mean_ap import MeanAveragePrecision

#### Utility functions

##### Function: clean_folders

In [2]:
# Clean all files before starting the process
def clean_folders(path, file_typ, fname_wildcard = None):
    folder_path = path
    if fname_wildcard is None:
        files = glob.glob(os.path.join(folder_path, f"*.{file_typ}"))
    else:
        files = glob.glob(os.path.join(folder_path, f"*{fname_wildcard}*.{file_typ}"))

    if files:
        for file_path in files:
            try:
                os.remove(file_path)                
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")
        #print(f"Deleted files in: {folder_path}")
            


##### Function: get_frame_names

In [3]:
# Get frame names
def get_frame_names(source_video_frame_dir):
    # scan all the JPEG frame names in this directory
    frame_names = [
        p for p in os.listdir(source_video_frame_dir)
        if os.path.splitext(p)[-1] in [".jpg", ".jpeg", ".JPG", ".JPEG"]
    ]
    #frame_names.sort(key=lambda p: int(os.path.splitext(p)[0]))
    frame_names.sort(key=lambda p: os.path.splitext(p)[0])
    return frame_names

##### Function: expand_box

In [4]:
def expand_box(box, image_width, image_height, margin_ratio=0.05):
    """
    Expands a bounding box by a margin ratio (e.g. 0.05 = 5%)
    """
    x_min, y_min, x_max, y_max = box
    box_width = x_max - x_min
    box_height = y_max - y_min

    # Compute margin in pixels
    margin_x = box_width * margin_ratio
    margin_y = box_height * margin_ratio

    # Expand box with clipping to image boundaries
    x_min_new = max(0, x_min - margin_x) 
    y_min_new = max(0, y_min - margin_y) 
    x_max_new = min(image_width, x_max + margin_x) 
    y_max_new = min(image_height, y_max + margin_y) 

    return [x_min_new, y_min_new, x_max_new, y_max_new]

##### Function: show_mask, show_points, show_box

In [5]:
def show_mask(mask, ax, obj_id=None, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        cmap = plt.get_cmap("tab10")
        cmap_idx = 0 if obj_id is None else obj_id
        color = np.array([*cmap(cmap_idx)[:3], 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_points(coords, labels, ax, marker_size=200):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)


def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))

##### Function: crop_image

In [6]:
# crop all the images within the boxes
def crop_image(image_path, output_path, box):
    """
    Crop a part of the image based on the given (x1, y1, x2, y2) coordinates.

    :param image_path: Path to the input image.
    :param output_path: Path to save the cropped image.
    :param box: Tuple (x1, y1, x2, y2) defining the crop area.
    """
    with Image.open(image_path) as img:
        cropped_img = img.crop(box)
        #cropped_img.show()  # Show the cropped image
        #print(cropped_img)
        #print(box)
        #cropped_img.save(output_path)

    return cropped_img

##### Function: crop_image_based_on_mask_and_save

In [7]:
def crop_image_based_on_mask_and_save(mask,img_path,output_path):

    mask = np.squeeze(mask)
    # Load image
    image_source, image = load_image(img_path)
    image_source_np = image_source.copy()
    
    # Ensure mask is binary (0 or 255) and convert to uint8
    mask_unit8 = (mask * 255).astype(np.uint8)  # Convert float32 to uint8

    # Find non-zero pixels in the mask (object region)
    coords = np.column_stack(np.where(mask_unit8 > 0))    
    #print(coords.shape[1])
    #print(coords)
    
    # Get bounding box (x_min, y_min) -> (x_max, y_max)
    y_min, x_min  = coords.min(axis=0)
    y_max, x_max  = coords.max(axis=0)    

    # Crop the image and mask using the bounding box
    cropped_image = image_source_np[y_min:y_max, x_min:x_max]
    cropped_mask = mask_unit8[y_min:y_max, x_min:x_max]

    # Convert the image to RGBA (adds an alpha channel for transparency)
    cropped_image = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2BGRA)
    
    # Set the background to transparent where the mask is 0
    cropped_image[:, :, 3] = cropped_mask  # Alpha channel is set to mask values

    # Convert to PIL Image and save
    cropped_pil = Image.fromarray(cropped_image)

    # Check with VLM. Is it same as prompt ?
        # Ensure image is in correct format
    if cropped_pil.mode != "RGB":
        cropped_pil = cropped_pil.convert("RGB")

    # Resize the image 
    cropped_pil = cropped_pil.resize((224, 224))
    #cropped_pil.show()

    # Save the cropped image 
    cropped_pil.save(output_path)

##### Function: show_annotated_frame

In [8]:
def show_annotated_frame(img_path,ann_frame_idx,boxes,confidences,class_ids,labels,filter_type,masks=None,):
    """
    Visualize image with supervision API
    """
    img = cv2.imread(img_path)
    
    if masks is not None:
        detections = sv.Detections(
            xyxy=boxes,
            mask=masks.astype(bool),
            confidence=np.array(confidences),
            class_id=class_ids
        )
    else:
        detections = sv.Detections(
            xyxy=boxes,
            confidence=np.array(confidences),
            class_id=class_ids
        )
    
    box_annotator = sv.BoxAnnotator()
    annotated_frame = box_annotator.annotate(scene=img.copy(), detections=detections)
    
    label_annotator = sv.LabelAnnotator()
    annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
    #cv2.imwrite(os.path.join(save_tracking_results_dir, "groundingdino_annotated_image.jpg"), annotated_frame)
    
    if masks is not None:
        mask_annotator = sv.MaskAnnotator()
        annotated_frame = mask_annotator.annotate(scene=annotated_frame, detections=detections)
        #cv2.imwrite(os.path.join(save_tracking_results_dir, "grounded_sam2_annotated_image_with_mask.jpg"), annotated_frame)
     
    plt.figure(figsize=(10, 10))
    plt.title(f"frame {ann_frame_idx} - {filter_type}")
    plt.imshow(annotated_frame)
    #plt.axis("off")
    plt.show()
    plt.close("all")

##### Function: filter_boxes_based_area_sw

In [9]:
def filter_boxes_based_area_sw(image_height, image_width, boxes, logits, phrases, box_area_threshold = 0.5 ):
    filtered_boxes = []
    filtered_logits = []
    filtered_phrases = []

    
    for i in range(len(boxes)):
        box = boxes[i]
        logit = logits[i]
        phrase = phrases[i]
        #print(f'h:{image_height}, w:{image_width}, th: {box_area_threshold}, boxes {box}')
        x_min, y_min, x_max, y_max = box         
        box_area = (x_max - x_min) * (y_max - y_min)
        image_area = image_height * image_width
        #print(f'h:{image_height}, w:{image_width}, th: {box_area_threshold}, boxes: {box}, b area: {box_area}, i area: {image_area} ')
        if box_area < box_area_threshold * image_area:  # keep only smaller boxes
            #print(f'boxes: {box} - Filtered')
            filtered_boxes.append(box)
            filtered_logits.append(logit)
            filtered_phrases.append(phrase)

    return filtered_boxes, filtered_logits, filtered_phrases

##### Function: normalize_boxes, scale_boxes

In [10]:
def normalize_boxes(boxes, width, height):
    return [
        [box[0] / width, box[1] / height, box[2] / width, box[3] / height]
        for box in boxes
    ]


def scale_boxes(boxes, width, height):
    return [
        [int(x1 * width), int(y1 * height), int(x2 * width), int(y2 * height)]
        for x1, y1, x2, y2 in boxes
    ]

##### Function: sliding_window

In [11]:
def sliding_window(image, window_size=(512, 512), stride=256):
    image_width, image_height = image.size
    window_width, window_height = window_size
    crops = []
    img_copy = image.copy()
    image_copy_np = np.array(img_copy)
   
    # Derive the x and y start cordinates
    x_starts = list(range(0, image_width - window_width + 1, stride))
    y_starts = list(range(0, image_height - window_height + 1, stride))

    # Ensure right edge is covered
    if x_starts[-1] + window_width < image_width:
        x_starts.append(image_width - window_width)

    # Ensure bottom edge is covered
    if y_starts[-1] + window_height < image_height:
        y_starts.append(image_height - window_height)
    
    for y in y_starts:
        for x in x_starts:            
            crop = image.crop((x, y, x + window_width, y + window_height))
            crops.append((crop, (x, y)))
            top_left = (x, y)
            bottom_right = (x + window_width, y + window_height)
            color = tuple(np.random.randint(0, 255, size=3).tolist()) # random color for each window
            cv2.rectangle(image_copy_np, top_left, bottom_right, color=color, thickness=4)

    return crops, image_copy_np

##### Function: run_grounding_dino_on_window

In [12]:
def run_grounding_dino_on_window(crop, model, prompt, window_idx, box_threshold=0.30, text_threshold=0.70, box_area_threshold_window = 0.1):
    # Convert crop (PIL) to expected format
    crop_np = np.array(crop)
    crop_tensor = torchvision.transforms.ToTensor()(crop).cuda()

    # Run prediction (adapt predict function if needed for crops)
    boxes, logits, phrases = predict(
        model=model,
        image=crop_tensor,
        caption=prompt,
        box_threshold=box_threshold,
        text_threshold=text_threshold
    )

    # process the boxes        
    crop_width, crop_height = crop.size
    boxes = boxes * torch.Tensor([crop_width, crop_height, crop_width, crop_height])
    input_boxes = box_convert(boxes=boxes, in_fmt="cxcywh", out_fmt="xyxy").numpy()
    
    
    filtered_boxes, filtered_logits, filtered_phrases = filter_boxes_based_area_sw(crop_height, 
                                                                                   crop_width, 
                                                                                   input_boxes, 
                                                                                   logits, 
                                                                                   phrases, 
                                                                                   box_area_threshold = box_area_threshold_window)
    
    #if len(filtered_boxes) > 0:
    #    plot_annotate(crop_np, filtered_boxes, filtered_logits, filtered_phrases, window_idx)
    
    #return boxes.cpu().numpy(),logits, phrases
    return filtered_boxes,filtered_logits, filtered_phrases


##### Function: compute_box_center_weight

In [13]:
def compute_box_center_weight(box, window_width, window_height):
    box_center_x = (box[0] + box[2]) / 2
    box_center_y = (box[1] + box[3]) / 2
    win_center_x = window_width / 2
    win_center_y = window_height / 2

    dx = abs(box_center_x - win_center_x) / window_width
    dy = abs(box_center_y - win_center_y) / window_height

    # Closer to center → weight near 1, further → penalized to 0.5 or lower
    distance = (dx ** 2 + dy ** 2) ** 0.5  # Euclidean
    weight = max(0.5, 1.0 - distance * 2)  # Adjust factor if needed
    return round(weight, 2)


##### Function: deduplicate_using_wbf

In [14]:
def deduplicate_using_wbf(all_boxes, all_confidences, all_labels, all_weights, width, height, wbf_iou_thr, wbf_skip_box_thr, wbf_conf_type):
    # Deduplicating boxes using wbf
    all_boxes_norm = normalize_boxes(all_boxes, width, height)

    all_boxes_norm = [
                        [min(max(coord, 0), 1) for coord in box]  # clamp each coordinate between zero and one
                        for box in all_boxes_norm                # for each box
                    ]

    all_boxes_norm_fin = [all_boxes_norm]               # list of boxes  
    scores_list    = [[float(conf) for conf in all_confidences]] # list of N floats  
    labels_list    = [[int(lbl) for lbl in all_labels]]          # list of N ints  
    
    if all_weights is None:
        weights_list = None
    else:    
        weights_list   = [float(w) for w in all_weights]          # optional per-box weights  

    fused_boxes, unique_confidences, unique_labels = weighted_boxes_fusion(
                                #all_boxes_norm_fin, all_confidences, all_labels,
                                all_boxes_norm_fin, scores_list, labels_list,
                                #iou_thr=0.05, skip_box_thr=0.05
                                weights=weights_list,iou_thr=wbf_iou_thr, skip_box_thr=wbf_skip_box_thr, conf_type=wbf_conf_type
                            )
    unique_boxes = scale_boxes(fused_boxes, width, height)

    return unique_boxes, unique_confidences, unique_labels

##### Function: deduplicate_using_nms

In [15]:
def deduplicate_using_nms(all_boxes, all_confidences, all_labels):
    # Deduplicating boxes using nms
    all_boxes_tensor = torch.tensor(all_boxes, dtype=torch.float32) # (N, 4)
    all_confidences_tensor = torch.tensor(all_confidences, dtype=torch.float32) # (N,)
    indices = nms(all_boxes_tensor, all_confidences_tensor, iou_threshold=0.0)

    unique_boxes = [all_boxes[i] for i in indices]
    unique_confidences = [all_confidences[i] for i in indices]
    unique_labels = [all_labels[i] for i in indices]
    
    return unique_boxes, unique_confidences, unique_labels

##### Function: infer_large_image

In [16]:
#def infer_large_image(image_path, prompt, model,display,ann_frame_idx, window_size=(512, 512), stride=256): # nms
def infer_large_image(image_path, 
                      prompt, 
                      model,
                      display,
                      vis_sliding_window,
                      vis_original_gd_boxes,
                      vis_gd_boxes_after_deduplication,
                      ann_frame_idx,
                      height,
                      width, 
                      window_size=(512, 512), 
                      stride=256,
                      box_area_threshold_window = 0.1,
                      gd_box_threshold=0.30, 
                      gd_text_threshold=0.70,
                      de_duplicate_boxes_method="wbf",
                      wbf_iou_thr=0.6,
                      wbf_skip_box_thr=0.20,
                      wbf_conf_type="max"): #wbf
    
    window_width, window_height = window_size
    image = Image.open(image_path).convert("RGB")
    
    """
    # Increasing the size of the image.. but then whole pipeline needs to be adjuested to accomadate the larger size.
    image = image.resize((int(image.width * 1.5), int(image.height * 1.5)),
                         resample=Image.BICUBIC
                        )
    """
    windows, image_with_windows = sliding_window(image, window_size, stride)

    if display and vis_sliding_window:
        # Display the windows on the image
        plt.figure(figsize=(10, 10))
        plt.imshow(image_with_windows)
        #plt.axis("off")
        plt.title("Sliding Window Visualization")
        plt.show()
        plt.close("all")

    all_boxes = []
    all_confidences = []
    all_labels = []
    all_weights = []
    window_idx = 0
    class_ids =  []
    unique_boxes = []
    unique_confidences = []
    unique_labels = []

    for crop, (offset_x, offset_y) in tqdm(windows, desc="Running windowed inference", unit="window", leave=False):
    #for crop, (offset_x, offset_y) in windows:
        window_idx += 1
        #print(window_idx)
        #print(crop.size)
        
        boxes, confidences, labels = run_grounding_dino_on_window(crop, 
                                                                  model, 
                                                                  prompt, 
                                                                  window_idx, 
                                                                  gd_box_threshold, 
                                                                  gd_text_threshold, 
                                                                  box_area_threshold_window)

        if len(boxes) > 0:
            #plot_annotate(crop,boxes,confidences,labels,window_idx)
            class_ids = np.array(list(range(len(labels))))
            #show_annotated_frame(crop,window_idx,boxes,confidences,class_ids,labels)
    
            # Adjust coordinates to global image space
            #for box in boxes:
            for i in range(len(boxes)):
                box = boxes[i]
                confidence = confidences[i].item()
                label = labels[i]
                
                x_min, y_min, x_max, y_max = box
                global_box = [
                    x_min.item() + offset_x,
                    y_min.item() + offset_y,
                    x_max.item() + offset_x,
                    y_max.item() + offset_y,
                ]
                
                weight = compute_box_center_weight(global_box, window_width, window_height)

                all_boxes.append(global_box)
                all_confidences.append(float(confidence))
                #all_confidences.append(round(confidence,2))
                all_labels.append("0")
                all_weights.append(float(weight))
           

    # filter out the duplicate boxes, Run NMS (IoU threshold ~0.5–0.6 recommended)
    if len(all_boxes) > 0:
        
        #print(all_boxes)
        #print(all_confidences)
        #print(all_labels)
        #print(all_confidences.shape)
        #print(len(all_boxes), len(all_confidences), len(all_labels))
        #print([len(b) for b in all_boxes])        # per-model box count
        #print([len(c) for c in all_confidences])  # per-model score count
        #print([len(l) for l in all_labels])       # per-model label count
        
        
        # Visualise origianl gd boxes
        if display and vis_original_gd_boxes:
            all_class_ids = np.array(list(range(len(all_boxes))))            
            
            labels_dis = [
                f"Car:{confidence:.2f}"
                for confidence
                in all_confidences
            ]
            #print(f'GD WBF confidences : {unique_confidences_rounded}')
            show_annotated_frame(image_path, ann_frame_idx, np.array(all_boxes), all_confidences, all_class_ids, labels_dis, "GD Boxes - original (before deduplication)", None)
        
        
        if de_duplicate_boxes_method == 'wbf':
            # Deduplicating boxes using nms
            unique_boxes, unique_confidences, unique_labels = deduplicate_using_wbf(all_boxes, all_confidences, all_labels, all_weights, width, height, wbf_iou_thr, wbf_skip_box_thr, wbf_conf_type)
        elif de_duplicate_boxes_method == 'nms':            
            # Deduplicating boxes using nms
            unique_boxes, unique_confidences, unique_labels = deduplicate_using_nms(all_boxes, all_confidences, all_labels)
            
        #print(f'UB: {unique_boxes}')
        #print(f'UC: {unique_confidences}')
        #print(f'UL: {unique_labels}')
        #print(all_confidences.shape)
        #return np.array(all_boxes), all_confidences, all_labels
        #print(unique_boxes[0])
        #print(len(unique_boxes[0]))
        if display and vis_gd_boxes_after_deduplication and len(unique_boxes) > 0:
            unique_class_ids = np.array(list(range(len(unique_boxes))))            
            #string_labels = [str(l[0]) if isinstance(l, (list, tuple)) else str(l) for l in unique_labels]
            unique_confidences_rounded = [np.float64(round(confidence, 2)) for confidence in unique_confidences]
            string_labels = [
                f"Car:{confidence:.2f}"
                for confidence
                in unique_confidences_rounded
            ]
            #print(f'GD WBF confidences : {unique_confidences_rounded}')
            
            if de_duplicate_boxes_method == 'wbf':
                frame_title = "GD Boxes - after wbf"  
            elif  de_duplicate_boxes_method == 'nms':
                frame_title = "GD Boxes - after nms"  

            show_annotated_frame(image_path, ann_frame_idx, np.array(unique_boxes), unique_confidences_rounded, unique_class_ids, string_labels, frame_title, None)
           
            
    
    return unique_boxes, unique_confidences, unique_labels

##### Function: get_unique_object_ids

In [17]:
def get_unique_object_ids(path_cropped_images,output_path_csv, resnet50_model, obj_similarity_threshold, df):
    if df is None:
        # read csv
        df = pd.pandas    
        df = pd.read_csv(output_path_csv, sep=";")

    tracked_objects = {}  # obj_id -> embedding
    next_obj_id = 0
    assigned_ids = []
    cropped_image_list = df["cropped_image_name"]

    # Preprocessing transform
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),  # ResNet's expected input
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],  # Standard for ImageNet-trained ResNet
                            std=[0.229, 0.224, 0.225])
    ])

    for cropped_image_name in cropped_image_list:
        crop_path = os.path.join(path_cropped_images, cropped_image_name)
        img = Image.open(crop_path).convert("RGB")
        input_tensor = preprocess(img).unsqueeze(0)
        with torch.no_grad():
            emb = resnet50_model(input_tensor).squeeze().numpy()
    

        best_sim = 0
        best_id = None

        for obj_id, prev_emb in tracked_objects.items():
            sim = cosine_similarity([emb], [prev_emb])[0][0]
            #if sim > best_sim and sim > 0.8645:  # similarity threshold
            #if sim > best_sim and sim > 0.7:  # similarity threshold, all objects are same id
            #if sim > best_sim and sim > 0.82:  # similarity threshold
            #if sim > best_sim and sim > 0.73:  # similarity threshold
            if sim > best_sim and sim > obj_similarity_threshold:  # similarity threshold
                best_sim = sim
                best_id = obj_id

        if best_id is not None:
            assigned_ids.append(best_id)
        else:
            tracked_objects[next_obj_id] = emb
            assigned_ids.append(next_obj_id)
            next_obj_id += 1
    df["obj_ids"] = assigned_ids

    # Save to CSV back    
    df.to_csv(output_path_csv, index=False, sep=";")

##### Function: center_box, add_new_points_for_sam2_vediosegmentation, add_new_boxes_for_sam2_vediosegmentation

In [18]:
"""
Register each object's positive points to video predictor with seperate add_new_points call
"""
def center_box(box):
    x_center = y_center = 0.0    
    x_min, y_min, x_max, y_max = box
    x_center = (x_max + x_min) / 2
    y_center = (y_max + y_min) / 2

    return [x_center, y_center]

def add_new_points_for_sam2_vediosegmentation(df_inp
                                             ,video_predictor
                                             ,inference_state
                                             ,source_video_frame_dir
                                             ,vis_sam2_vedio_prompts
                                             ,vis_sam2_vedio_prompts_num_frames):
    df = df_inp.sort_values(by=["obj_ids","box_gd_confidence"],ascending=[True,False])
    frame_indexes = df["frame_id"]    
    box_coords = df["box_coord"]
    #box_coords = ast.literal_eval(df["box_coord"])
    box_labels = df["box_label"]
    object_ids = df["obj_ids"]
    image_names = df["image_name"]
    confidences = df["box_gd_confidence"]
    #object_id = 1
    processed_obj_ids =  []
    # for labels, `1` means positive click and `0` means negative click
    labels = np.array([1], np.int32)
    objids_with_confidences = {}
    vis_cnt = 0

    for object_id, ann_frame_idx, label, box, image_name, confidence in zip(object_ids,frame_indexes, box_labels, box_coords,image_names,confidences):
    #for object_id, (ann_frame_idx, label, box) in enumerate(zip(frame_indexes, box_labels, box_coords), start=1):
    #for _, (ann_frame_idx, label, box) in enumerate(zip(frame_indexes, box_labels, box_coords), start=1):
        if object_id not in processed_obj_ids:
            processed_obj_ids.append(object_id) #only process an object id once
            objids_with_confidences[object_id] = confidence
            box = ast.literal_eval(box)
            box_center_points = center_box(box)# get the center of the box
            box_center_points_np = np.array([box_center_points], dtype=np.float32)

            #print(f'object_id: {object_id}')            
            #print(f'ann_frame_idx: {ann_frame_idx}')
            #print(f'box_center_points_np: {box_center_points_np}')
            #print(f'labels: {labels}')

            #box = [",".join([f"{x:.5f}" for x in box])]
            _, out_obj_ids, out_mask_logits = video_predictor.add_new_points_or_box(
                inference_state=inference_state,
                frame_idx=ann_frame_idx,
                obj_id=object_id,
                points=box_center_points_np,
                labels=labels,
            )

            
            # uncomment when needed
            # show the results on the current (interacted) frame
            #print("Displaying point prompts for SAM2 video tracking using unique obkects across frame")
            if vis_sam2_vedio_prompts and vis_cnt < vis_sam2_vedio_prompts_num_frames:             
                plt.figure(figsize=(9, 6))
                plt.title(f"{image_name}_frameid_{ann_frame_idx}_uniqueObjectID_{object_id}")
                plt.imshow(Image.open(os.path.join(source_video_frame_dir, image_name)))
                show_points(box_center_points_np, labels, plt.gca())
                show_mask((out_mask_logits[0] > 0.0).cpu().numpy(), plt.gca(), obj_id=out_obj_ids[0])
                plt.show()
                plt.close("all")
                vis_cnt += 1
            
            
    return objids_with_confidences


"""
Register each object's positive boxes to video predictor with seperate add_new_points call
"""
"""
def add_new_boxes_for_sam2_vediosegmentation(df):
    frame_indexes = df["frame_id"]    
    box_coords = df["box_coord"]
    #box_coords = ast.literal_eval(df["box_coord"])
    box_labels = df["box_label"]
    object_ids = df["obj_ids"]
    #object_id = 1
    processed_obj_ids =  []

    for object_id, ann_frame_idx, label, box in zip(object_ids,frame_indexes, box_labels, box_coords):
    #for object_id, (ann_frame_idx, label, box) in enumerate(zip(frame_indexes, box_labels, box_coords), start=1):
    #for _, (ann_frame_idx, label, box) in enumerate(zip(frame_indexes, box_labels, box_coords), start=1):
        if object_id not in processed_obj_ids:
            processed_obj_ids.append(object_id) #only process an object id once
            box = ast.literal_eval(box)
            #box = [",".join([f"{x:.5f}" for x in box])]
            _, out_obj_ids, out_mask_logits = video_predictor.add_new_points_or_box(
                inference_state=inference_state,
                frame_idx=ann_frame_idx,
                obj_id=object_id,
                box=box,
            )
"""

'\ndef add_new_boxes_for_sam2_vediosegmentation(df):\n    frame_indexes = df["frame_id"]    \n    box_coords = df["box_coord"]\n    #box_coords = ast.literal_eval(df["box_coord"])\n    box_labels = df["box_label"]\n    object_ids = df["obj_ids"]\n    #object_id = 1\n    processed_obj_ids =  []\n\n    for object_id, ann_frame_idx, label, box in zip(object_ids,frame_indexes, box_labels, box_coords):\n    #for object_id, (ann_frame_idx, label, box) in enumerate(zip(frame_indexes, box_labels, box_coords), start=1):\n    #for _, (ann_frame_idx, label, box) in enumerate(zip(frame_indexes, box_labels, box_coords), start=1):\n        if object_id not in processed_obj_ids:\n            processed_obj_ids.append(object_id) #only process an object id once\n            box = ast.literal_eval(box)\n            #box = [",".join([f"{x:.5f}" for x in box])]\n            _, out_obj_ids, out_mask_logits = video_predictor.add_new_points_or_box(\n                inference_state=inference_state,\n          

##### Function: extract_box_from_mask

In [19]:
def extract_box_from_mask(mask, threshold=0.5, min_area=20, max_area = 4500):
    """
    Extracts bounding box from a binary mask.
    - `mask`: float mask [0–1]
    - `threshold`: binarization threshold
    - `min_area`: minimum number of pixels required to consider a valid object
    """
    # Ensure 2D shape
    if mask.ndim > 2:
        mask = np.squeeze(mask)

    binary_mask = (mask > threshold).astype(np.uint8)

    # Remove small noise via connected components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    if num_labels <= 1:
        return None  # No object found

    # Take the largest connected component (skip background - label 0)
    largest_comp_idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    filtered_mask = (labels == largest_comp_idx).astype(np.uint8)

    coords = np.column_stack(np.where(filtered_mask > 0))
    if coords.shape[0] < min_area or coords.shape[0] > max_area:
        return None  # Too small or big to be real car

    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)

    return [x_min, y_min, x_max, y_max]


##### Function: load_predictions_from_csv, load_ground_truth_from_csv

In [20]:
# --------- Utility Functions ---------
def load_predictions_from_csv(csv_path, csv_path_gt):
    df = pd.read_csv(csv_path, sep = ';')    
    predictions = []
    images_in_prediction = []

    for img_id, group in df.groupby("image_name"):
        images_in_prediction.append(img_id)
        #mask_str = ','.join(list(map(str, mask_unit8.flatten())))
        #print(mask_str)
        #boxes = torch.tensor([clean_box_string(b) for b in group["box_coord"]], dtype=torch.float32)
        #boxes = torch.tensor([','.join(list(ast.literal_eval(b) for b in group["box_coord"]))], dtype=torch.float32)
        #boxes = torch.tensor(group["box_coord"].values, dtype=torch.float32)
        boxes = torch.tensor(group[["x_min", "y_min", "x_max", "y_max"]].values, dtype=torch.float32)
        scores = torch.tensor(group["box_gd_confidence"].values, dtype=torch.float32)
        labels = torch.tensor(group["class_id"].values, dtype=torch.int64)

        predictions.append({
            "image_name": img_id,
            "boxes": boxes,
            "scores": scores,
            "labels": labels
        })

    df_gt = pd.read_csv(csv_path_gt, sep = ';')
    image_names_gt = []
    image_names_gt = df_gt["image_name"].unique()

    for image_name_gt in image_names_gt:
        if image_name_gt not in images_in_prediction:
            predictions.append({
                "image_name": image_name_gt,
                "boxes": torch.empty((0, 4), dtype=torch.float32),
                "scores": torch.tensor([], dtype=torch.float32),
                "labels": torch.tensor([], dtype=torch.int64)
            })


    return predictions


def load_ground_truth_from_csv(csv_path):
    df = pd.read_csv(csv_path, sep = ';')
    #print(df.columns)
    #print(df.head())
    ground_truth = []

    for img_id, group in df.groupby("image_name"):
        boxes = torch.tensor(group[["x_min", "y_min", "x_max", "y_max"]].values, dtype=torch.float32)
        labels = torch.tensor(group["class_id"].values, dtype=torch.int64)

        ground_truth.append({
            "image_name": img_id,
            "boxes": boxes,
            "labels": labels
        })

    return ground_truth

##### Function: draw_boxes

In [21]:
# Function to draw boxes
def draw_boxes(img, boxes, color, label):
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

##### Function: visualise_pred_and_groundtruth

In [22]:
def visualise_pred_and_groundtruth(img_folder, pred_csv, gt_csv,vis_num_frames):

    # Load CSVs
    pred_df = pd.read_csv(pred_csv, sep = ';')
    gt_df = pd.read_csv(gt_csv, sep = ';')

    # Visualize few samples
    for image_name in gt_df['image_name'].unique()[:vis_num_frames]:  # only first 5 images
        img_path = os.path.join(img_folder, image_name)
        image = cv2.imread(img_path)

        gt_boxes = gt_df[gt_df['image_name'] == image_name][["x_min", "y_min", "x_max", "y_max"]].values
        pred_boxes = pred_df[pred_df['image_name'] == image_name][["x_min", "y_min", "x_max", "y_max"]].values

        draw_boxes(image, gt_boxes, (0, 255, 0), "GT")      # Green for ground truth
        draw_boxes(image, pred_boxes, (255, 0, 0), "Pred")  # Blue for predictions

        # Convert BGR to RGB for matplotlib
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(10, 8))
        plt.imshow(image_rgb)
        plt.title(f"Image: {image_name}")
        #plt.axis("off")
        plt.show()
        plt.close("all")

##### Function: result_metrics_visualise

In [23]:
def result_metrics_visualise(save_tracking_results_dir
                             ,model_preds_csv_fname
                             ,ground_truth_csv_fname
                             ,layer
                             ,source_video_frame_dir
                             ,vis_pred_and_gt
                             ,vis_pred_and_gt_num_frames):

    model_preds_csv = os.path.join(save_tracking_results_dir, model_preds_csv_fname)
    ground_truth_csv = os.path.join(save_tracking_results_dir, ground_truth_csv_fname)

    #  Load Data ---------
    predictions = load_predictions_from_csv(model_preds_csv, ground_truth_csv)
    ground_truth = load_ground_truth_from_csv(ground_truth_csv)

    #  Evaluate  ---------
    metric = MeanAveragePrecision()
    metric.update(predictions, ground_truth)
    metrics = metric.compute()

    #  Print Results ---------
    print(f"\nModel Metrics: {layer}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")  
  
    # Visualise ground truth and predicted boxes  
    if vis_pred_and_gt == True:
        print(f"\nVisualise GT and Pred boxes: {layer}")
        visualise_pred_and_groundtruth(source_video_frame_dir, model_preds_csv, ground_truth_csv, vis_pred_and_gt_num_frames)
    
    return metrics

##### Function: create_box_list

In [24]:
# Get dataframe, returns box cordinates as a list
def create_box_list(row):
    return [row['x_min'], row['y_min'], row['x_max'], row['y_max']]

### Pipeline
- GroundingDIno + Pre-filter using Gemma3
- if good result from pre-filter then:
-       SAM2 + Post-filter using Gemma3
-       If good result from post-filter then:
-         Save the cropped image
-         Save the tracking result in csv

In [25]:
def pipeline_4layers(frame_names
                    ,num_frames_to_process
                    ,source_video_frame_dir
                    ,save_tracking_results_dir
                    ,path_cropped_images
                    ,path_cropped_images_final_layer
                    ,path_masks
                    ,path_masks_final_layer
                    ,DEVICE
                                        
                    ,grounding_model                # Inputs for grounding dino     
                    ,gd_text_prompt                 
                    ,gd_box_threshold
                    ,gd_text_threshold                    
                    
                    ,window_size                     # Inuts needed for sliding windo
                    ,stride
                    ,box_area_threshold_window
                    ,de_duplicate_boxes_method
                    ,wbf_iou_thr
                    ,wbf_skip_box_thr
                    ,wbf_conf_type
                    
                    ,expand_box_margin_ratio_org  # margin_ratio to expand original boxes coming from grounding dino. This is to keep the contextual information for the VLM's use.
                    ,expand_box_margin_ratio_out  # margin_ratio to expand final boxes out of the pipeline. Boxes out of fifth layer are very tight as they are created based on sam2 generated segments
                    
                    ,vlm_model                    # FLag that tell which vlm to use, plaigemma2 or gemma3
                    ,gemma3_model                   # VLM/filter
                    ,gemma3_processor  
                    ,gemma3_prompt_pre_filter              
                    ,gemma3_prompt_post_filter 
                    ,gemma3_system_content_prompt
                    ,paligemma2_model
                    ,paligemma2_processor
                    ,paligemma2_prompt_pre_filter
                    ,paligemma2_prompt_post_filter
                    ,expected_answer 

                    ,sam2_predictor                 # sam2 
                    ,resnet50_model                 # resnet50 for finding unique object ids
                    ,obj_similarity_threshold 
                    ,sam2_video_box_thr_for_prop    # Objects having box threshold hgher than this value will be propogated using sam2 video propogation 

                    ,video_predictor

                    ,vis_1layer_pred_and_gt
                    ,vis_1layer_pred_and_gt_num_frames 
                    ,vis_2layer_pred_and_gt
                    ,vis_2layer_pred_and_gt_num_frames 
                    ,vis_4layer_pred_and_gt
                    ,vis_4layer_pred_and_gt_num_frames                                                                                   
                    ,vis_5layer_pred_and_gt
                    ,vis_5layer_pred_and_gt_num_frames
                    ,vis_sam2_vedio_prompts
                    ,vis_sam2_vedio_prompts_num_frames
                    ,vis_5layer_after_wbf_pred_and_gt
                    ,vis_5layer_after_wbf_pred_and_gt_num_frames 
                    
                    ,vis_frame_stride               # Visualisation stride
                    ,vis_sliding_window             # Sliding Window Visualization
                    ,vis_original_gd_boxes          # Visualise origianl gd boxes
                    ,vis_gd_boxes_after_deduplication         # GD Boxes - after wbf
                    ,vis_result_after_pre_filter    # Result after Pre-filter
                    ,vis_result_after_post_filter   # Result after Post-filter
                    ,vis_sam2_vedio_out_mask

                    ,execute_1_4_Layers             # True if only 1-4 layers are to be executed
                    ,execute_5Layer                 # Set to True/False if 5 layer (SAM2 video propogation) should be executed/not executed
                    ,
                    ):

    tracking_result = {}
    tracking_result_1Layer = {}
    tracking_result_2Layer = {}    
    tracking_result_4Layer = {}
    tracking_result_5Layer = {}
    prev_ann_frame_idx = None
    indices = np.array    
    display = None
    x_min_exp =  y_min_exp = x_max_exp = y_max_exp = 0

    if num_frames_to_process == 0:
        num_frames_to_process = len(frame_names)


    if execute_1_4_Layers == True:
        # For each frame
        for ann_frame_idx in tqdm(range(num_frames_to_process),desc = 'Tracking', unit = 'frame'):
            
            if vis_frame_stride > 0:
                if ann_frame_idx % vis_frame_stride == 0:
                    display = True
                else:
                    display = False
            
            frame_good_boxes_pf_id = 0 # Variable to store id of good boxes (i.e. after post filter) per frame

            #if num_frames_to_process > 0 and ann_frame_idx > num_frames_to_process:
            #    break
                
            # prompt grounding dino to get the box coordinates on specific frame
            img_path = os.path.join(source_video_frame_dir, frame_names[ann_frame_idx])
            image_source, image = load_image(img_path)
            h, w, _ = image_source.shape

            ##########################################################
            ###               GroundingDINO                        ###
            ##########################################################
            
            # Predict from groundingDino
            boxes, confidences, labels = infer_large_image(img_path 
                                                        ,gd_text_prompt 
                                                        ,grounding_model
                                                        ,display
                                                        ,vis_sliding_window
                                                        ,vis_original_gd_boxes
                                                        ,vis_gd_boxes_after_deduplication
                                                        ,ann_frame_idx 
                                                        ,h, w 
                                                        ,window_size 
                                                        ,stride
                                                        ,box_area_threshold_window
                                                        ,gd_box_threshold 
                                                        ,gd_text_threshold
                                                        ,de_duplicate_boxes_method
                                                        ,wbf_iou_thr
                                                        ,wbf_skip_box_thr
                                                        ,wbf_conf_type)  #with wbf
            
            # Save result from first layer - grounding dino
            key = frame_names[ann_frame_idx][:-4]
            tracking_result_1Layer[key] = {}
            tracking_result_1Layer[key]["frame_id"] = ann_frame_idx
            tracking_result_1Layer[key]["image_name"] = frame_names[ann_frame_idx]
            
            if len(boxes) > 0:
                box_id = 0
                for box_id, (box, confidence, label) in enumerate(zip(boxes, confidences, labels)):
                    x_min, y_min, x_max, y_max = box
                    tracking_result_1Layer[key][box_id] = {}
                    tracking_result_1Layer[key][box_id]["box_id"] = box_id
                    tracking_result_1Layer[key][box_id]["box_coord"] = box
                    tracking_result_1Layer[key][box_id]["box_label"] = label
                    tracking_result_1Layer[key][box_id]["box_gd_confidence"] = round(confidence,2)                
                    tracking_result_1Layer[key][box_id]["x_min"] = x_min
                    tracking_result_1Layer[key][box_id]["y_min"] = y_min
                    tracking_result_1Layer[key][box_id]["x_max"] = x_max
                    tracking_result_1Layer[key][box_id]["y_max"] = y_max
            

            # Discard boxes that are greater than  box_area_threshold times of the total image (aprox 64*64 pixel)    
            input_boxes, confidences, labels = filter_boxes_based_area_sw(h, w, boxes, confidences, labels, box_area_threshold = 0.002)  
            #input_boxes, confidences, labels = is_car_shape(boxes, confidences, labels) # Does not work so well.
            
            # Add margin to all boxes coming from grounding dino by margin_ratio %. 
            # This is so that the subsequent layers (judge) get to see a bit of background for the contextual knowledge.
            expanded_boxes = [
                #expand_box(box, w, h, margin_ratio=0.15) # map_50: 0.0137
                expand_box(box, w, h, margin_ratio=expand_box_margin_ratio_org)
                for box in input_boxes
            ]
            input_boxes = np.array(expanded_boxes)
            #print(input_boxes)
            #print(expanded_boxes)

            image_path = img_path  # image path
            good_boxes = []
            good_labels = []
            good_confidences = []

            ##########################################################
            ###           Pre-filter using Gemma3                  ###
            ##########################################################    
            
            # Prepare for savie results from second layer.
            tracking_result_2Layer[key] = {}
            tracking_result_2Layer[key]["frame_id"] = ann_frame_idx
            tracking_result_2Layer[key]["image_name"] = frame_names[ann_frame_idx]

            for i in range(len(input_boxes)):
                #cropped_image_name = f'{frame_names[ann_frame_idx]}_{i}'
                cropped_image_name = f'{i}.jpeg'
                output_path = os.path.join(save_tracking_results_dir, cropped_image_name)
                box = input_boxes[i]
                #box = np.array([boxes[i][-2], boxes[i][-1], boxes[i][0],boxes[i][1]])  # box coordinates (x1, y1, x2, y2) from groundingDino
                image_cropped = crop_image(image_path, output_path, box)
            
                # Convert tensor to PIL Image correctly
                if isinstance(image_cropped, torch.Tensor):
                    # Move tensor to CPU, remove batch dim if exists, convert to uint8
                    image_cropped = image_cropped.squeeze(0).permute(1, 2, 0).cpu().numpy()  # Convert CHW -> HWC
                    image_cropped = (image_cropped * 255).astype(np.uint8)  # Scale to 0-255
                    image_cropped = Image.fromarray(image_cropped)  # Convert to PIL Image
            
                # Ensure image is in correct format
                if image_cropped.mode != "RGB":
                    image_cropped = image_cropped.convert("RGB")

                if vlm_model == "paligemma2":
                
                    # Resize the image as per the paligemma need
                    image_cropped = image_cropped.resize((224, 224))

                    # Set the inputs to the paligemma model
                    model_inputs = paligemma2_processor(text=paligemma2_prompt_pre_filter, images=image_cropped, return_tensors="pt").to(torch.bfloat16).to(paligemma2_model.device)
                    input_len = model_inputs["input_ids"].shape[-1]
                
                    
                    #### Using VLM (paligemma2) as a judge to check if the boxes generated by the groundingDino was actually the object mentioned in the prompt
                    # get the inference from the paligemma model
                    with torch.inference_mode():
                        #generation = model.generate(**model_inputs, max_new_tokens=20, do_sample=True, temperature=0.9, top_p=0.95)
                        generation = paligemma2_model.generate(**model_inputs, max_new_tokens=5, do_sample=False)
                        generation = generation[0][input_len:]
                        decoded = paligemma2_processor.decode(generation, skip_special_tokens=True)
            
                else:
                    if vlm_model == "gemma3":
                        
                        # Resize the image as per the gemma3 need
                        image_cropped = image_cropped.resize((896, 896))

                        messages = [
                            {
                                "role": "system",
                                "content": [{"type": "text", "text": gemma3_system_content_prompt}]  #gemma3_system_content_prompt
                                #"content": [{"type": "text", "text": "You are a helpful assistant."}]
                            },
                            {
                                "role": "user",
                                "content": [
                                    {"type": "image", "image": image_cropped},
                                    {"type": "text", "text": gemma3_prompt_pre_filter}
                                ]
                            }
                        ]


                        model_inputs = gemma3_processor.apply_chat_template(
                            messages, add_generation_prompt=True, tokenize=True,
                            return_dict=True, return_tensors="pt"
                        ).to(gemma3_model.device, dtype=torch.bfloat16)

                        input_len = model_inputs["input_ids"].shape[-1]

                        # max_new_tokens - Max new tokens to be generated by VLM (Gemma3, Paligemma2). 
                        # Since the final answer that we expect is Yes or No, 5 is a okay number. 
                        # More the number here, more computation is involved.
                        with torch.inference_mode():
                            generation = gemma3_model.generate(**model_inputs, max_new_tokens=5, do_sample=False, top_p = 1.0, top_k = 50)
                            generation = generation[0][input_len:]
                            decoded = gemma3_processor.decode(generation, skip_special_tokens=True)
                            
                            #if display:
                            #    print(f'Pre-filter - {decoded}')
            
                if decoded == expected_answer:
                    # Save the boxes that paligemma says are the same as prompt
                    good_boxes.append(box)
                    good_labels.append(labels[i])
                    #good_confidences.append(round(confidences[i].cpu().numpy().tolist(),2))
                    good_confidences.append(round(confidences[i],2))
            
                    
            good_boxes = np.array(good_boxes)        

            # Save result from second layer - pre-filter (VLM - Gemma3)               
            if len(good_boxes) > 0:
                box_id = 0
                for box_id, (box, confidence, label) in enumerate(zip(good_boxes, good_confidences, good_labels)):
                    x_min, y_min, x_max, y_max = box
                    tracking_result_2Layer[key][box_id] = {}
                    tracking_result_2Layer[key][box_id]["box_id"] = box_id
                    tracking_result_2Layer[key][box_id]["box_coord"] = box
                    tracking_result_2Layer[key][box_id]["box_label"] = label
                    tracking_result_2Layer[key][box_id]["box_gd_confidence"] = round(confidence,2)                
                    tracking_result_2Layer[key][box_id]["x_min"] = x_min
                    tracking_result_2Layer[key][box_id]["y_min"] = y_min
                    tracking_result_2Layer[key][box_id]["x_max"] = x_max
                    tracking_result_2Layer[key][box_id]["y_max"] = y_max
            
            
            # Show the images with the good boxes after the pre-filter
            if len(good_boxes) > 0:
                class_ids = np.array(list(range(len(good_labels))))

                labels = [
                    f"Car: {confidence:.2f}"
                    for confidence in good_confidences                
                ]
                
                if display and vis_result_after_pre_filter:
                    #print(f'Pre filter good_boxes       : {good_boxes}')
                    #print(f'Pre filter good_labels      : {good_labels}')
                    #print(f'Pre filter good_confidences : {good_confidences}')
                    show_annotated_frame(img_path,ann_frame_idx,good_boxes,good_confidences,class_ids,labels,"Result after Pre-filter",None)


            ##########################################################
            ###           SAM2 - Image segmentation                ###
            ##########################################################
            
            # If there are good boxes then continue with next step (i.e. SAM2 segmentation + Post filter). Otherwise exit from here and continue to next image
            if len(good_boxes) > 0:
                
                #### Use the good boxes coming out of VLM judgement as input to SAM2-image for segmentation
                image_source_np = image_source.copy()
                #print(image_source_np.shape, image_source_np.dtype)
            
                sam2_predictor.set_image(image_source_np)
            
                masks, scores, logits = sam2_predictor.predict(
                    point_coords=None,
                    point_labels=None,
                    box=good_boxes,
                    multimask_output=False,
                )

                # convert the shape of the masks to (n, H, W)
                if masks.ndim == 4:
                    masks = masks.squeeze(1)
                #print(len(masks))
            
                good_boxes_pf = []
                good_labels_pf = []
                good_confidences_pf = []
                good_masks_pf = []
                
                
                ##########################################################
                ###           Post-filter using Gemma3                 ###
                ########################################################## 
                
                # Loop through all the masks
                for i in range(len(masks)):
                    
                    # Ensure mask is binary (0 or 255) and convert to uint8
                    mask_unit8 = (masks[i] * 255).astype(np.uint8)  # Convert float32 to uint8
                
                    # Find non-zero pixels in the mask (object region)
                    coords = np.column_stack(np.where(mask_unit8 > 0))
                
                    
                    """
                    # Get bounding box (x_min, y_min) -> (x_max, y_max)
                    y_min, x_min  = coords.min(axis=0)
                    y_max, x_max  = coords.max(axis=0)  
                    """
                    if coords.size == 0:
                        continue # Skip “no coordinates”
                    else:
                        # Get bounding box (x_min, y_min) -> (x_max, y_max)
                        y_min, x_min  = coords.min(axis=0)
                        y_max, x_max  = coords.max(axis=0)  

                    sam2_box = [x_min, y_min, x_max, y_max]
                    #print(sam2_box)
                    # What happens if we expand the boxes here.. Does post filter wok better?
                    #x_min_exp, y_min_exp, x_max_exp, y_max_exp = expand_box(sam2_box, w, h, margin_ratio=0.30) 

                    #print(f'[{x_min_exp}, {y_min_exp}, {x_max_exp}, {y_max_exp}]')

                                
                    # Crop the image and mask using the bounding box
                    #cropped_image = image_source_np[int(y_min_exp):int(y_max_exp), int(x_min_exp):int(x_max_exp)]
                    #cropped_mask = mask_unit8[int(y_min_exp):int(y_max_exp), int(x_min_exp):int(x_max_exp)]


                    cropped_image = image_source_np[y_min:y_max, x_min:x_max]   # Crop the image as per sam2 bounding box
                    cropped_mask = mask_unit8[y_min:y_max, x_min:x_max]  #crop mask, not needed here
            
                    # Convert the image to RGBA (adds an alpha channel for transparency)
                    cropped_image = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2BGRA)
                    
                    # Set the background to transparent where the mask is 0
                    #cropped_image[:, :, 3] = cropped_mask  # Alpha channel is set to mask values
                
                    # Convert to PIL Image and save
                    cropped_pil = Image.fromarray(cropped_image)
            
                    # Check with VLM. Is it same as prompt ?
                        # Ensure image is in correct format
                    if cropped_pil.mode != "RGB":
                        cropped_pil = cropped_pil.convert("RGB")
                

                
                    if vlm_model == "paligemma2":

                        # Resize the image as per the paligemma need
                        cropped_pil = cropped_pil.resize((224, 224))
                        #cropped_pil_rgb.show()
                    
                        # Set the inputs to the plaigemma model
                        model_inputs = paligemma2_processor(text=paligemma2_prompt_post_filter, images=cropped_pil, return_tensors="pt").to(torch.bfloat16).to(paligemma2_model.device)
                        input_len = model_inputs["input_ids"].shape[-1]
                    
                        # get the inferece from the paligemma model
                        with torch.inference_mode():
                            #generation = model.generate(**model_inputs, max_new_tokens=20, do_sample=True, temperature=0.9, top_p=0.95)
                            generation = paligemma2_model.generate(**model_inputs, max_new_tokens=5, do_sample=False)
                            generation = generation[0][input_len:]
                            decoded = paligemma2_processor.decode(generation, skip_special_tokens=True)
                            #if display:
                            #    print(f'Post-filter - {decoded}')
                    
                    else:
                        if vlm_model == "gemma3":

                            # Resize the image as per the gemma3 need
                            cropped_pil = cropped_pil.resize((896, 896))
                            #cropped_pil_rgb.show()

                            messages = [
                                {
                                    "role": "system",
                                    "content": [{"type": "text", "text": gemma3_system_content_prompt}]
                                    #"content": [{"type": "text", "text": "You are a helpful assistant."}]
                                },
                                {
                                    "role": "user",
                                    "content": [
                                        {"type": "image", "image": cropped_pil},
                                        {"type": "text", "text": gemma3_prompt_post_filter}
                                    ]
                                }
                            ]


                            model_inputs = gemma3_processor.apply_chat_template(
                                messages, add_generation_prompt=True, tokenize=True,
                                return_dict=True, return_tensors="pt"
                            ).to(gemma3_model.device, dtype=torch.bfloat16)

                            input_len = model_inputs["input_ids"].shape[-1]

                            with torch.inference_mode():
                                generation = gemma3_model.generate(**model_inputs, max_new_tokens=5, do_sample=False, top_p = 1.0, top_k = 50)
                                generation = generation[0][input_len:]
                                decoded = gemma3_processor.decode(generation, skip_special_tokens=True)
                                
                                #if display:
                                #    print(f'Post-filter - {decoded}')

                    if decoded == expected_answer:
                        # Save the boxes that paligemma says are the same as prompt
                        good_boxes_pf.append(good_boxes[i])
                        good_labels_pf.append(good_labels[i])
                        rounded_good_confidences = round(good_confidences[i],2)
                        good_confidences_pf.append(rounded_good_confidences)
                        good_masks_pf.append(masks[i])

                        # Convert the masks to list of string so as to save it in excel
                        #print(mask_unit8)
                        #print('------------')
                        
                        #print(mask_unit8.shape)
                        #mask_str = ','.join(list(map(str, mask_unit8.flatten())))
                        #print(mask_str)

                    
                        # Save the cropped images (cropped based on segments from SAM2)
                        frame_good_boxes_pf_id += 1                
                        id_name = f'{frame_names[ann_frame_idx][:-4]}_{ann_frame_idx}_{frame_good_boxes_pf_id}'
                        cropped_image_name = f'{id_name}.jpeg'
                        output_path = os.path.join(path_cropped_images, cropped_image_name)
                        cropped_image_sam2 = image_source_np[y_min:y_max, x_min:x_max]   # Crop the image as per sam2 bounding box
                        cropped_pil_sam2 = Image.fromarray(cropped_image_sam2)    # Convert to PIL Image and save
                        #cropped_mask = mask_unit8[y_min:y_max, x_min:x_max]  #crop mask, not needed here
                        cropped_pil_sam2.save(output_path)

                        # Save the indices of the non-zero masks
                        indices = np.argwhere(mask_unit8 > 0)
                        indices_str = '/'.join([f'{r},{c}' for r, c in indices])
                        indices_str = indices_str.replace("\n","").replace("\t","")
                        mask_file_name = f'{id_name}.npy'
                        output_path_mask = os.path.join(path_masks, mask_file_name)
                        #np.save(output_path_mask, indices_str)

                        # Save the tracking results
                        if ann_frame_idx != prev_ann_frame_idx:                
                            key = frame_names[ann_frame_idx][:-4]
                            tracking_result_4Layer[key] = {}
                            tracking_result_4Layer[key]["image_name"] = frame_names[ann_frame_idx]
                            tracking_result_4Layer[key]["frame_id"] = ann_frame_idx
                            tracking_result_4Layer[key][frame_good_boxes_pf_id] = {}
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["box_id"] = frame_good_boxes_pf_id
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["box_coord"] = good_boxes[i].tolist()
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["box_label"] = good_labels[i]
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["box_gd_confidence"] = rounded_good_confidences
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["mask_file_name"] = mask_file_name
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["cropped_image_name"] = cropped_image_name
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["x_min"] = x_min
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["y_min"] = y_min
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["x_max"] = x_max
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["y_max"] = y_max
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["x_min_exp"] = x_min_exp
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["y_min_exp"] = y_min_exp
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["x_max_exp"] = x_max_exp
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["y_max_exp"] = y_max_exp                    

                            prev_ann_frame_idx = ann_frame_idx
                        else:
                            tracking_result_4Layer[key][frame_good_boxes_pf_id] = {}
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["box_id"] = frame_good_boxes_pf_id
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["box_coord"] = good_boxes[i].tolist()
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["box_label"] = good_labels[i]
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["box_gd_confidence"] = rounded_good_confidences
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["mask_file_name"] = mask_file_name
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["cropped_image_name"] = cropped_image_name
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["x_min"] = x_min
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["y_min"] = y_min
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["x_max"] = x_max
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["y_max"] = y_max
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["x_min_exp"] = x_min_exp
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["y_min_exp"] = y_min_exp
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["x_max_exp"] = x_max_exp
                            tracking_result_4Layer[key][frame_good_boxes_pf_id]["y_max_exp"] = y_max_exp 

                            
                good_boxes_pf = np.array(good_boxes_pf)  
                good_masks_pf = np.array(good_masks_pf)
                
                # Show the images with the final boxes and maske after the post filter
                if len(good_boxes_pf) > 0:
                    class_ids_pf = np.array(list(range(len(good_labels_pf))))
            
                    labels_pf = [
                        f"Car:{confidence:.2f}"
                        for confidence in good_confidences_pf                    
                    ]


                    if display and vis_result_after_post_filter:
                        #print(f'Post filter good_boxes       : {good_boxes_pf}')
                        #print(f'Post filter good_labels      : {good_labels_pf}')
                        #print(f'Post filter good_confidences : {good_confidences_pf}')
                        #show_annotated_frame(img_path,ann_frame_idx,good_boxes,good_confidences,class_ids,labels,"Result after Pre-filter",None)
                        show_annotated_frame(img_path,ann_frame_idx,good_boxes_pf,good_confidences_pf,class_ids_pf,labels_pf,"Result after Post-filter",good_masks_pf)

        
        #########################################################################################
        ####        Save the tracking result of first layer in csv                           ####
        #########################################################################################

        #     
        rows = []
        for image_id, data in tracking_result_1Layer.items():
            image_name = data["image_name"]
            frame_id = data["frame_id"]

            for box_id, box in data.items():
                if isinstance(box_id, int):
                    rows.append({
                        "image_id": image_id,
                        "image_name": image_name,
                        "frame_id": frame_id,
                        "box_id": box_id,
                        "box_coord": box["box_coord"],
                        "box_label": box["box_label"],
                        "box_gd_confidence": box["box_gd_confidence"],
                        "x_min" : box["x_min"],
                        "y_min" : box["y_min"],
                        "x_max" : box["x_max"],
                        "y_max" : box["y_max"],
                        "class_id": 0
                    })

        # Create a DataFrame
        df = pd.DataFrame(rows)

        # Save to CSV
        output_path_csv = os.path.join(save_tracking_results_dir, "tracking_result_1Layer.csv")
        df.to_csv(output_path_csv, index=False, sep=";",quoting=1)

        #########################################################################################
        ####  Save the tracking result of second layer (Pre-filter - VLM - Gemma3) in csv    ####
        #########################################################################################

        #    
        rows = []
        for image_id, data in tracking_result_2Layer.items():
            image_name = data["image_name"]
            frame_id = data["frame_id"]

            for box_id, box in data.items():
                if isinstance(box_id, int):
                    rows.append({
                        "image_id": image_id,
                        "image_name": image_name,
                        "frame_id": frame_id,
                        "box_id": box_id,
                        "box_coord": box["box_coord"],
                        "box_label": box["box_label"],
                        "box_gd_confidence": box["box_gd_confidence"],
                        "x_min" : box["x_min"],
                        "y_min" : box["y_min"],
                        "x_max" : box["x_max"],
                        "y_max" : box["y_max"],
                        "class_id": 0
                    })

        # Create a DataFrame
        df = pd.DataFrame(rows)

        # Save to CSV
        output_path_csv = os.path.join(save_tracking_results_dir, "tracking_result_2Layer.csv")
        df.to_csv(output_path_csv, index=False, sep=";",quoting=1)

        #########################################################################################
        ####  Save the tracking result of fourth layer (Post-filter - VLM - Gemma3) in csv   ####
        #########################################################################################

        # Save the tracking result of fourth layer in csv    
        rows = []
        for image_id, data in tracking_result_4Layer.items():
            image_name = data["image_name"]
            frame_id = data["frame_id"]

            for box_id, box in data.items():
                if isinstance(box_id, int):
                    rows.append({
                        "image_id": image_id,
                        "image_name": image_name,
                        "frame_id": frame_id,
                        "box_id": box_id,
                        "box_coord": box["box_coord"],
                        "box_label": box["box_label"],
                        "box_gd_confidence": box["box_gd_confidence"],
                        "mask_file_name": box["mask_file_name"],
                        "cropped_image_name": box["cropped_image_name"],
                        "x_min" : box["x_min"],
                        "y_min" : box["y_min"],
                        "x_max" : box["x_max"],
                        "y_max" : box["y_max"],
                        "x_min_exp" : box["x_min_exp"],
                        "y_min_exp" : box["y_min_exp"],
                        "x_max_exp" : box["x_max_exp"],
                        "y_max_exp" : box["y_max_exp"],
                        "class_id": 0,
                        "confidence": 1.0 
                    })

        # Create a DataFrame
        df = pd.DataFrame(rows)

        # Save to CSV
        output_path_csv = os.path.join(save_tracking_results_dir, "tracking_result_4Layer.csv")
        df.to_csv(output_path_csv, index=False, sep=";",quoting=1)


        #########################################################################################
        ####       Delete the models to free up space in GPU                                 ####
        #########################################################################################

        # 
        del grounding_model
        del sam2_predictor
        del gemma3_model
        del gemma3_processor
        
        gc.collect()   # Run Python's garbage collector to drop unreferenced objects
        torch.cuda.empty_cache()

    #########################################################################################
    ####          Get unique object ids using resnet and cosine similarity               ####
    #########################################################################################
    if execute_5Layer == True:  
        # Find and save the unique object ids 
        output_path_csv = os.path.join(save_tracking_results_dir, "tracking_result_4Layer.csv")

        # get_unique_object_ids also updates tracking_result_4Layer.csv with the unique object id
        get_unique_object_ids(path_cropped_images, output_path_csv, resnet50_model, obj_similarity_threshold, None)  # output_path_csv path to tracking_result_4Layer.csv
        del resnet50_model  
        gc.collect()   # Run Python's garbage collector to drop unreferenced objects  
        torch.cuda.empty_cache()

        #########################################################################################
        ####                           SAM2 - video tracking                                ####
        #########################################################################################
        
        #torch.autocast(device_type=DEVICE, dtype=torch.bfloat16).__enter__() # Note: figure how does this influence the G-DINO model..bfloat16 is not okay for groundingdino 
        with torch.autocast(device_type=DEVICE, dtype=torch.bfloat16):
            inference_state = video_predictor.init_state(video_path=source_video_frame_dir) # init video predictor state
            
            df_4Layer = pd.pandas    
            df_4Layer = pd.read_csv(output_path_csv, sep=";") # Read tracking_result_4Layer.csv which also has the unique object ids now
            
            # Seperate high and low confidence rows. Only high confidence detections will be propogated
            df_4Layer_high_conf = df_4Layer[df_4Layer["box_gd_confidence"] >= sam2_video_box_thr_for_prop].copy()
            df_4Layer_low_conf = df_4Layer[df_4Layer["box_gd_confidence"] < sam2_video_box_thr_for_prop].copy()
            
            df_sam2_Video = df_4Layer_high_conf    # Propogate only high confidence boxes
                  
            objids_with_confidences = add_new_points_for_sam2_vediosegmentation(df_sam2_Video
                                                                                ,video_predictor
                                                                                ,inference_state,source_video_frame_dir 
                                                                                ,vis_sam2_vedio_prompts
                                                                                ,vis_sam2_vedio_prompts_num_frames) 

            
            #Propagate the video predictor to get the segmentation results for each frame
            video_segments = {}  # video_segments contains the per-frame segmentation results
            for out_frame_idx, out_obj_ids, out_mask_logits in video_predictor.propagate_in_video(inference_state,start_frame_idx=0):
                # Clear memory before processing new frame
                torch.cuda.empty_cache()
                gc.collect()
                
                video_segments[out_frame_idx] = {
                    out_obj_id: (out_mask_logits[i] > 0.0).to(torch.uint8).cpu().numpy()
                    for i, out_obj_id in enumerate(out_obj_ids)
                }

        
        # render the segmentation results every few frames given by vis_frame_stride    
        plt.close("all")
        if vis_frame_stride > 0 and vis_sam2_vedio_out_mask:
            for out_frame_idx in range(0, len(frame_names), vis_frame_stride):
                plt.figure(figsize=(6, 4))
                plt.title(f"{frame_names[out_frame_idx]}_frameid_{out_frame_idx}")
                plt.imshow(Image.open(os.path.join(source_video_frame_dir, frame_names[out_frame_idx])))
                for out_obj_id, out_mask in video_segments[out_frame_idx].items():
                    show_mask(out_mask, plt.gca(), obj_id=out_obj_id)
                plt.show()
                plt.close("all")
            
        #layer5_image_box_confidence_label = {}
        # Save the results from the propogation
        frame_stride = 1
        rows_final = []
        
        
        for out_frame_idx in tqdm(range(0, len(frame_names), frame_stride),desc='Saving tracking result', unit='frame'):
        
            img_path = os.path.join(source_video_frame_dir, frame_names[out_frame_idx])
            #print("hello1")
            #plt.figure(figsize=(6, 4))
            #plt.title(f"frame {out_frame_idx}")
            #plt.imshow(Image.open(os.path.join(source_video_frame_dir, frame_names[out_frame_idx])))
            
            current_frame_has_mask = False

            #layer5_image_box_confidence_label[frame_names[out_frame_idx]] = {}
            #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["frame_id"] = out_frame_idx
            #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["boxes"] = []
            #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["confidences"] = []
            #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["labels"] = []
            #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["objectids"] = []

            for out_obj_id, out_mask in video_segments[out_frame_idx].items():            
                #show_mask(out_mask, plt.gca(), obj_id=out_obj_id)
                if np.all(out_mask == 0):
                    continue
                else:
                    box_final = extract_box_from_mask(out_mask)
                    if box_final is not None:
                        image_source, _ = load_image(img_path)
                        image_height, image_width, _ = image_source.shape            
                        x_min, y_min, x_max, y_max = expand_box(box_final, image_width, image_height, margin_ratio=expand_box_margin_ratio_out)

                        #box_fi = [x_min, y_min, x_max, y_max]
                        #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["boxes"].append(box_fi)
                        #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["confidences"].append(objids_with_confidences[out_obj_id])
                        #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["labels"].append(0)
                        #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["objectids"].append(out_obj_id)

                        # crop and save the cropped image                    
                        id_name = f'{frame_names[out_frame_idx][:-4]}_{out_frame_idx}_{out_obj_id}'
                        cropped_image_name_fl = f'{id_name}.jpeg'
                        output_path = os.path.join(path_cropped_images_final_layer, cropped_image_name_fl)
                        #crop_image_based_on_mask_and_save(out_mask,img_path,output_path)
                                            
                        # Save the indices of the non-zero masks
                        indices = np.argwhere(np.squeeze(out_mask) > 0)
                        #print(indices.shape)
                        #print(indices)            
                        indices_str = '/'.join([f'{r},{c}' for r, c in indices])
                        indices_str = indices_str.replace("\n","").replace("\t","")
                        mask_file_name = f'{id_name}.npy'
                        output_path_mask = os.path.join(path_masks_final_layer, mask_file_name)
                        #np.save(output_path_mask, indices_str)

                        rows_final.append({
                                "image_name": frame_names[out_frame_idx],
                                "frame_id": out_frame_idx,
                                "object_id": out_obj_id,                    
                                "x_min": x_min,
                                "y_min": y_min,
                                "x_max": x_max,
                                "y_max": y_max,
                                "mask_file_name": mask_file_name,
                                "cropped_image_name": cropped_image_name_fl,
                                "class_id": 0,
                                "box_gd_confidence": objids_with_confidences[out_obj_id]
                                })
                        
                        current_frame_has_mask = True
                
            if not current_frame_has_mask:
                #box_fi = [0, 0, 0, 0]
                #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["boxes"].append(box_fi)
                #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["confidences"].append(0.0)
                #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["labels"].append(0)
                #layer5_image_box_confidence_label[frame_names[out_frame_idx]]["objectids"].append(None)
                
                rows_final.append({
                        "image_name": frame_names[out_frame_idx],
                        "frame_id": out_frame_idx,
                        "object_id": out_obj_id,                    
                        "x_min": 0,
                        "y_min": 0,
                        "x_max": 0,
                        "y_max": 0,
                        "mask_file_name": None,
                        "cropped_image_name": None,
                        "class_id": 0,
                        "box_gd_confidence": 0.0
                    }) 

        # Create a DataFrame
        df_rows_final = pd.DataFrame(rows_final)
        df_4Layer_low_confidences = df_4Layer_low_conf[["image_name","frame_id", "obj_ids", "x_min", "y_min", "x_max", "y_max", "mask_file_name","cropped_image_name", "class_id", "box_gd_confidence"]]
        df_4Layer_low_confidences = df_4Layer_low_confidences.rename(columns={"obj_ids": "object_id"}) # rename the obj_ids column name to object_id
        df_final = pd.concat([df_rows_final,df_4Layer_low_confidences], ignore_index=True)   # concatenate the results from sam2 video tracking and the low confidence detection from 4th layer     

        # Save the final tracking result to CSV
        output_path_csv = os.path.join(save_tracking_results_dir, "tracking_result_5Layer.csv")  
        df_final.to_csv(output_path_csv, index=False, sep=";")

        
    ################################################
    # De-duplicate the detections from fifth layer #
    ################################################
    
    #df_final_5Layer = df_final.copy()
    path_csv_5Layer = os.path.join(save_tracking_results_dir, "tracking_result_5Layer.csv")
    df_final_5Layer = pd.read_csv(path_csv_5Layer, sep = ';')

    # Apply create box list to create new column which has box cordaintes as list
    df_final_5Layer["box"] = df_final_5Layer.apply(create_box_list, axis=1)

    # Group by the image name and argregate boxes, confidence and laels as list for each image
    df_final_5Layer_grouped = df_final_5Layer.groupby("image_name").agg({
            "frame_id" : "first"
        ,"box" : list
        ,"box_gd_confidence" : list
        ,"class_id" : list
    })

    df_final_5Layer_grouped = df_final_5Layer_grouped.rename(columns={"box": "boxes", "box_gd_confidence": "confidences", "class_id" : "labels" })

    rows_final_after_wbf = []
    
    for image_name, frame_data in df_final_5Layer_grouped.iterrows():
    #for image_name, frame_data in layer5_image_box_confidence_label.items():
        #image_name = frame_data["image_name"]
        frame_id = frame_data["frame_id"]
        boxes = frame_data["boxes"]
        confidences = frame_data["confidences"]
        labels = frame_data["labels"]      
                
        img_path = os.path.join(source_video_frame_dir, image_name)
        image_source, image = load_image(img_path)
        image_height, image_width, _ = image_source.shape

        unique_boxes, unique_confidences, unique_labels = [], [], []

        if len(boxes) > 0:
            unique_boxes, unique_confidences, unique_labels = deduplicate_using_wbf(boxes, confidences, labels, None, image_width, image_height, wbf_iou_thr, wbf_skip_box_thr, wbf_conf_type)
        else:
            unique_boxes, unique_confidences, unique_labels = boxes, confidences, labels
        
        for box, confidence, label in zip(unique_boxes, unique_confidences, unique_labels):
            rows_final_after_wbf.append({
                "image_name": image_name,
                "frame_id": frame_id,                                    
                "x_min": box[0],
                "y_min": box[1],
                "x_max": box[2],
                "y_max": box[3],
                "class_id": 0,
                "box_gd_confidence": confidence
                })

    # Create a DataFrame
    df_rows_final_after_wbf = pd.DataFrame(rows_final_after_wbf)
    # Save the final tracking result to CSV
    output_path_csv = os.path.join(save_tracking_results_dir, "tracking_result_5Layer_after_wbf.csv")  
    df_rows_final_after_wbf.to_csv(output_path_csv, index=False, sep=";") 


    #########################################################################################
    ####                           RESULTS FROM EACH LAYER                               ####
    #########################################################################################

    # Initialise the metrics dict for each layer 
    metrics_1Layer, metrics_2Layer, metrics_4Layer, metrics_5Layer, metrics_5Layer_after_wbf = {}, {}, {}, {}, {}
    # Set the grount truth file name
    ground_truth_fname = "ground_truth.csv"
    
    
    #### Results at layer 1 GroundingDino  
    model_preds_fname = "tracking_result_1Layer.csv"
    layer = "Post layer 1 (GroundingDino)"
    metrics_1Layer = result_metrics_visualise(save_tracking_results_dir
                            ,model_preds_fname
                            ,ground_truth_fname
                            ,layer
                            ,source_video_frame_dir
                            ,vis_1layer_pred_and_gt
                            ,vis_1layer_pred_and_gt_num_frames)


    #### Results at layer 2 Pre-filter - VLM - Gemma3
    model_preds_fname = "tracking_result_2Layer.csv"
    layer = "Post layer 2 (Pre-filter - VLM - Gemma3)"
    metrics_2Layer = result_metrics_visualise(save_tracking_results_dir
                        ,model_preds_fname
                        ,ground_truth_fname
                        ,layer
                        ,source_video_frame_dir
                        ,vis_2layer_pred_and_gt
                        ,vis_2layer_pred_and_gt_num_frames)


    #### Results at last layer 4    
    model_preds_fname = "tracking_result_4Layer.csv"    
    layer = "Post layer 4 (Post-filter - VLM - Gemma3)"
    metrics_4Layer = result_metrics_visualise(save_tracking_results_dir
                    ,model_preds_fname
                    ,ground_truth_fname
                    ,layer
                    ,source_video_frame_dir
                    ,vis_4layer_pred_and_gt
                    ,vis_4layer_pred_and_gt_num_frames)

    #### Results at last layer 5 (sam2 video tracking)
    execute_5Layer = True
    if execute_5Layer == True:
        model_preds_fname = "tracking_result_5Layer.csv"
        layer = "Post layer 5 (SAM2 video tracking)"
        metrics_5Layer = result_metrics_visualise(save_tracking_results_dir
                    ,model_preds_fname
                    ,ground_truth_fname
                    ,layer
                    ,source_video_frame_dir
                    ,vis_5layer_pred_and_gt
                    ,vis_5layer_pred_and_gt_num_frames)
    
    #### Results at last layer 5 (sam2 video tracking) after wbf
    model_preds_fname = "tracking_result_5Layer_after_wbf.csv"
    layer = "Post layer 5 (SAM2 video tracking) - after wbf"
    metrics_5Layer_after_wbf = result_metrics_visualise(save_tracking_results_dir
                ,model_preds_fname
                ,ground_truth_fname
                ,layer
                ,source_video_frame_dir
                ,vis_5layer_after_wbf_pred_and_gt
                ,vis_5layer_after_wbf_pred_and_gt_num_frames)
    
    return metrics_1Layer, metrics_2Layer, metrics_4Layer, metrics_5Layer, metrics_5Layer_after_wbf
                 

## Execute the pipeline

In [ ]:
## A. Execute the pipeline  

# General setup
#cuda ="cuda:1"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
num_images = 100
experiment_name = "1_final"
source_video_frame_dir = f"/home/devsin-3/1. Master thesis/snowball/{num_images}/train-frames/jpg"
path_cropped_images = f"/home/devsin-3/1. Master thesis/snowball/{num_images}/{experiment_name}/cropped-images"
path_cropped_images_final_layer = f"/home/devsin-3/1. Master thesis/snowball/{num_images}/{experiment_name}/cropped-images/final-layer"
path_masks = f"/home/devsin-3/1. Master thesis/snowball/{num_images}/{experiment_name}/masks"
path_masks_final_layer = f"/home/devsin-3/1. Master thesis/snowball/{num_images}/{experiment_name}/masks/final-layer"
save_tracking_results_dir = f"/home/devsin-3/1. Master thesis/snowball/{num_images}/{experiment_name}/tracking-results"
output_video_path = f"/home/devsin-3/1. Master thesis/snowball/{num_images}/{experiment_name}/tracking-results/uav_car_tracking.mp4"

#Ground dino config
gd_config = "/home/devsin-3/GroundingDINO/groundingdino/config/GroundingDINO_SwinB_cfg.py"
gd_checkpoint = "/home/devsin-3/GroundingDINO/groundingdino/checkpoints/groundingdino_swinb_cogcoor.pth"


#Environment settings and model initialization for Grounding DINO
grounding_model = load_model(
    model_config_path=gd_config, 
    model_checkpoint_path=gd_checkpoint,
    device=DEVICE
)
#grounding_model = grounding_model.to(torch.float16)





# build SAM2 image predictor
sam2_checkpoint = "//home/devsin-3/thesis/lib/python3.10/site-packages/sam2/checkpoints/sam2.1_hiera_tiny.pt"
sam2_config  = "//home/devsin-3/thesis/lib/python3.10/site-packages/sam2/configs/sam2.1/sam2.1_hiera_t.yaml"
sam2_model = build_sam2(sam2_config, sam2_checkpoint, device=DEVICE)
sam2_predictor = SAM2ImagePredictor(sam2_model)

### Preparation for SAM2 vedio tracking
### Unique objects across frames needs to be added just once for SAM2 prompts. If its added more than once then SAM2 sees them as unique objects even though if they ar same. Therefore mark all the objects with unique object ids and then add propmpt to SAM2 only once for each unique ID.
### Here resnet50 model is used to extract embeddings of the cropped images.. these embeddings are compared to each other basedon cosine similarity. Objects that are highly similar are marked with same object id.
#resnet50_model = models.resnet50(pretrained=True) # Pretrained ResNet 
resnet50_model = models.resnet50(weights=ResNet50_Weights.DEFAULT) # Pretrained ResNet   
resnet50_model = torch.nn.Sequential(*list(resnet50_model.children())[:-1])  # Remove last classification layer
resnet50_model.eval()




# Pipeline parameters
#gd_box_thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45 ] # grounding dino 
#gd_box_thresholds = [0.05, 0.15, 0.25, 0.35, 0.45 ] # grounding dino 
#gd_box_thresholds = [0.05, 0.25, 0.45 ] # grounding dino 
gd_box_thresholds = [0.10] # grounding dino 
gd_text_threshold = 1.00
gd_text_prompt = "car ."
window_size=(384, 384) 
#window_size=(1920, 1080)
stride=320
box_area_threshold_window = 0.03
wbf_iou_thr=0.6
wbf_skip_box_thr=0.35  # This should be same as gd_box_threshold
wbf_conf_type="max"
expand_box_margin_ratio_org = 0.30  # margin_ratio to expand original boxes coming from grounding dino. This is to keep the contextual information for the VLM's use.
expand_box_margin_ratio_out = 0.20  # margin_ratio to expand final boxes out of the pipeline. Boxes out of fifth layer are very tight as they are created based on sam2 generated segments
num_frames_to_process = 0  #Use this to test small number of frames. If want to test with the whole dataset then set it to 0
obj_similarity_threshold = 0.65 # Used to assigned object ids to objects detected at fourth layer. Objects that are similar will be assigned same objet id.
vlm_model = "paligemma2"  # can be paligemma2 or gemma3
de_duplicate_boxes_method = "wbf"  # can choose between wbf and nms
sam2_video_box_thr_for_prop = 0.4    # Objects having box threshold higher than this value will be propogated using sam2 video propogation  
execute_1_4_Layers = True   # Set to true if layers 1-4 are to be excuted, else false
execute_5Layer = True       # Set to true if layer 5 is to be excuted, else false
clean_files = True          # Set to true if whole pipeline is to be run.. It will clean all the result files


vis_frame_stride = 5 # visualise results after each vis_frame_stride frames. Set to 0 if no visualization needed. This number is used for the following visualisation only
vis_sliding_window = False # Sliding Window Visualization
vis_original_gd_boxes = False # Visualise origianl gd boxes
vis_gd_boxes_after_deduplication = False # GD Boxes - after wbf
vis_result_after_pre_filter = False # Result after Pre-filter
vis_result_after_post_filter = False # Result after Post-filter
vis_sam2_vedio_out_mask = False # render the segmentation results at fifth layer every few frames


vis_1layer_pred_and_gt = False              # These are for visualising predicted vs ground truth boxes at each layer
vis_1layer_pred_and_gt_num_frames = 0       # set the number of frames to visualise  
vis_2layer_pred_and_gt = False
vis_2layer_pred_and_gt_num_frames = 50 
vis_4layer_pred_and_gt = False
vis_4layer_pred_and_gt_num_frames = 50                                                                                  
vis_5layer_pred_and_gt = False
vis_5layer_pred_and_gt_num_frames = 20
vis_sam2_vedio_prompts = False
vis_sam2_vedio_prompts_num_frames = 20
vis_5layer_after_wbf_pred_and_gt = False
vis_5layer_after_wbf_pred_and_gt_num_frames = 20


##################################################
#  Setup up the chosen VLM for acting as judge.  #
##################################################

# Initialise the setup
gemma3_model = None                    
gemma3_processor = None
gemma3_prompt_pre_filter = None
gemma3_prompt_post_filter = None
gemma3_system_content_prompt = None
paligemma2_model = None
paligemma2_processor = None
paligemma2_prompt_pre_filter = None
paligemma2_prompt_post_filter = None
expected_answer = None

if vlm_model == "gemma3":
    # Setup for the pre and post filter with VLM Gemma3
    gemma3_model_id = "google/gemma-3-4b-it"
    gemma3_model = Gemma3ForConditionalGeneration.from_pretrained(gemma3_model_id
                                                                ,device_map={"": "cuda:0"}
                                                                ,token = "hf_*").eval()
    gemma3_processor = AutoProcessor.from_pretrained(gemma3_model_id, use_fast=True, token = "hf_*")
    gemma3_prompt_pre_filter  = "This is a image crop of a drone-captured aerial image. Does it contain the top view of a car? Do not consider buildings, houses, shadows or any other objects. Only consider car. Answer strictly with either 'yes' or 'no' in lowercase."
    gemma3_prompt_post_filter = "This is a image segment of a drone-captured aerial image. Does it contain the top view of a car? Do not consider buildings, houses, shadows or any other objects. Only consider car. Answer strictly with either 'yes' or 'no' in lowercase."
    gemma3_system_content_prompt= "You are a vision-language expert. Identify cars in aerial drone images."
    expected_answer = 'yes'

    # All other prompts tried
    #prompt_pre_filter  = "The object in the image is not a car. Answer with 'yes' or 'no'."
    #prompt_post_filter = "The object in the image is not a car. Answer with 'yes' or 'no'."
    #prompt_pre_filter  = "The object in the image is a car. Answer with 'yes' or 'no'."
    #prompt_post_filter = "The object in the image is a car. Answer with 'yes' or 'no'."
    #prompt_pre_filter  = "This is a image crop of a drone-captured aerial image. Does it contain the top view of a car? Answer strictly with either 'yes' or 'no' in lowercase."
    #prompt_post_filter = "This is a image segment of a drone-captured aerial image. Does it contain the top view of a car? Answer strictly with either 'yes' or 'no' in lowercase."
else:
    if vlm_model == "paligemma2":
        
        # Setup for paligemma2
        paligemma2_model_id = "google/paligemma2-3b-pt-224"
        paligemma2_model = PaliGemmaForConditionalGeneration.from_pretrained(paligemma2_model_id, torch_dtype=torch.bfloat16, device_map={"": "cuda:0"}, token = "hf_*").eval()
        paligemma2_processor = PaliGemmaProcessor.from_pretrained(paligemma2_model_id, use_fast=True, token = "hf_*")
        paligemma2_prompt_pre_filter  = "<image> The object in the image is not a car. Answer with 'yes' or 'no'."
        paligemma2_prompt_post_filter = "<image> The object in the image is not a car. Answer with 'yes' or 'no'."
        expected_answer = 'no'

        #paligemma2_prompt_pre_filter  = "<image> The object in the image is a car. Answer with 'yes' or 'no'."
        #paligemma2_prompt_post_filter  = "<image> The object in the image is a car. Answer with 'yes' or 'no'."
        #expected_answer = 'yes'

        #prompt_post_filter = "<image> Is there a car in the image. Answer with 'yes' or 'no'."        
        #prompt_post_filter = "<image> Is there a car in the image. Answer with 'yes' or 'no'."
        #prompt_post_filter = "<image> Is this an image of a car. Answer with 'yes' or 'no'."


# get the frame names
frame_names = get_frame_names(source_video_frame_dir)
#print(len(frame_names))
#print(frame_names)



metrics_per_threshold = {}
for gd_box_threshold in tqdm(gd_box_thresholds, desc = "Executing pipeline", unit='gd box threshold'):
    
    if clean_files == True:
        # Clean all the files from previous runs
        clean_folders(path_masks,"npy")
        clean_folders(path_masks_final_layer,"npy")
        clean_folders(path_cropped_images,"jpeg")
        clean_folders(path_cropped_images_final_layer,"jpeg")
        clean_folders(save_tracking_results_dir, "csv", fname_wildcard = "Layer")

    ##### Use the good boxes coming out of Post-filter as input to SAM2-vedio for segmentation
    # Setup for sam video predictor and video predictor model
    video_predictor = build_sam2_video_predictor(sam2_config, sam2_checkpoint,device=DEVICE)
    video_predictor = video_predictor.to(torch.bfloat16)
    #print("Model dtype:", next(video_predictor.parameters()).dtype)
    #print("Device:", next(video_predictor.parameters()).device)

    wbf_skip_box_thr=gd_box_threshold  # Set wbf_skip_box_thr to gd_box_threshold. It should be same as gd_box_threshold

    # Call pipeline
    metrics_1Layer, metrics_2Layer, metrics_4Layer, metrics_5Layer, metrics_5Layer_after_wbf = pipeline_4layers(frame_names
                                                                                    ,num_frames_to_process
                                                                                    ,source_video_frame_dir
                                                                                    ,save_tracking_results_dir
                                                                                    ,path_cropped_images
                                                                                    ,path_cropped_images_final_layer
                                                                                    ,path_masks
                                                                                    ,path_masks_final_layer
                                                                                    ,DEVICE                                                                                    
                                                                                        
                                                                                    ,grounding_model                # Inputs for grounding dino     
                                                                                    ,gd_text_prompt                 
                                                                                    ,gd_box_threshold
                                                                                    ,gd_text_threshold                    
                                                                                        
                                                                                    ,window_size                     # Inuts needed for sliding windo
                                                                                    ,stride
                                                                                    ,box_area_threshold_window
                                                                                    ,de_duplicate_boxes_method
                                                                                    ,wbf_iou_thr
                                                                                    ,wbf_skip_box_thr
                                                                                    ,wbf_conf_type
                                                                                        
                                                                                    ,expand_box_margin_ratio_org  # margin_ratio to expand original boxes coming from grounding dino. This is to keep the contextual information for the VLM's use.
                                                                                    ,expand_box_margin_ratio_out  # margin_ratio to expand final boxes out of the pipeline. Boxes out of fifth layer are very tight as they are created based on sam2 generated segments                   
                                                                                        
                                                                                    ,vlm_model                    # FLag that tell which vlm to use, plaigemma2 or gemma3
                                                                                    ,gemma3_model                   # VLM/filter
                                                                                    ,gemma3_processor  
                                                                                    ,gemma3_prompt_pre_filter              
                                                                                    ,gemma3_prompt_post_filter 
                                                                                    ,gemma3_system_content_prompt
                                                                                    ,paligemma2_model
                                                                                    ,paligemma2_processor
                                                                                    ,paligemma2_prompt_pre_filter
                                                                                    ,paligemma2_prompt_post_filter
                                                                                    ,expected_answer 

                                                                                    ,sam2_predictor                 # sam2
                                                                                    ,resnet50_model                 # resnet50 for finding unique object ids
                                                                                    ,obj_similarity_threshold
                                                                                    ,sam2_video_box_thr_for_prop    # Objects having box threshold hgher than this value will be propogated using sam2 video propogation  

                                                                                    ,video_predictor

                                                                                    ,vis_1layer_pred_and_gt
                                                                                    ,vis_1layer_pred_and_gt_num_frames
                                                                                    ,vis_2layer_pred_and_gt
                                                                                    ,vis_2layer_pred_and_gt_num_frames 
                                                                                    ,vis_4layer_pred_and_gt
                                                                                    ,vis_4layer_pred_and_gt_num_frames                                                                                   
                                                                                    ,vis_5layer_pred_and_gt           
                                                                                    ,vis_5layer_pred_and_gt_num_frames
                                                                                    ,vis_sam2_vedio_prompts
                                                                                    ,vis_sam2_vedio_prompts_num_frames
                                                                                    ,vis_5layer_after_wbf_pred_and_gt
                                                                                    ,vis_5layer_after_wbf_pred_and_gt_num_frames  
                                                                                    
                                                                                    ,vis_frame_stride
                                                                                    ,vis_sliding_window             # Sliding Window Visualization
                                                                                    ,vis_original_gd_boxes          # Visualise origianl gd boxes
                                                                                    ,vis_gd_boxes_after_deduplication         # GD Boxes - after wbf
                                                                                    ,vis_result_after_pre_filter    # Result after Pre-filter
                                                                                    ,vis_result_after_post_filter   # Result after Post-filter
                                                                                    ,vis_sam2_vedio_out_mask        # Render the segmentation results at fifth layer every few frames

                                                                                    ,execute_1_4_Layers             # True if 1-4 layers to be executed, else false 
                                                                                    ,execute_5Layer                 # Set to False if 5 layer (SAM2 video propogation) should not be executed
                                                                                )

    metrics_per_threshold[gd_box_threshold] = {}
    metrics_per_threshold[gd_box_threshold]["1Layer"] = metrics_1Layer
    metrics_per_threshold[gd_box_threshold]["2Layer"] = metrics_2Layer
    metrics_per_threshold[gd_box_threshold]["4Layer"] = metrics_4Layer
    metrics_per_threshold[gd_box_threshold]["5Layer"] = metrics_5Layer
    metrics_per_threshold[gd_box_threshold]["5Layer_after_wbf"] = metrics_5Layer_after_wbf


# Get the best threshold
best_threshold = 0.0
best_map = 0.0
best_map50 = 0.0
best_map75 = 0.0
for threshold, metrics in metrics_per_threshold.items():
    execute_5Layer = True
    if execute_5Layer == True:
        if metrics["5Layer_after_wbf"]["map"] > best_map:
            best_threshold = threshold
            best_map  = metrics["5Layer_after_wbf"]["map"]
            best_map50  = metrics["5Layer_after_wbf"]["map_50"]
            best_map75  = metrics["5Layer_after_wbf"]["map_75"]
    else:
        if metrics["4Layer"]["map"] > best_map:
            best_threshold = threshold
            best_map  = metrics["4Layer"]["map"]
            best_map50  = metrics["4Layer"]["map_50"]
            best_map75  = metrics["4Layer"]["map_75"]


print(f'\nBest box threshold  : {best_threshold}')
print(f'Best map            : {best_map}')
print(f'Best map50          : {best_map50}')
print(f'Best map75          : {best_map75}')




final text_encoder_type: bert-base-uncased


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Executing pipeline:   0%|          | 0/1 [00:00<?, ?gd box threshold/s]

Tracking:   0%|          | 0/100 [00:00<?, ?frame/s]

Running windowed inference:   0%|          | 0/24 [00:00<?, ?window/s]

...

Running windowed inference:   0%|          | 0/24 [00:00<?, ?window/s]

propagate in video: 100%|██████████| 100/100 [00:42<00:00,  2.34it/s]


Saving tracking result:   0%|          | 0/100 [00:00<?, ?frame/s]


Model Metrics: Post layer 1 (GroundingDino)
map: 0.0437
map_50: 0.1833
map_75: 0.0043
map_small: 0.0098
map_medium: 0.0697
map_large: -1.0000
mar_1: 0.0345
mar_10: 0.1339
mar_100: 0.1984
mar_small: 0.1989
mar_medium: 0.1982
mar_large: -1.0000
map_per_class: -1.0000
mar_100_per_class: -1.0000
classes: 0.0000



Model Metrics: Post layer 2 (Pre-filter - VLM - Gemma3)
map: 0.0556
map_50: 0.1935
map_75: 0.0080
map_small: 0.0125
map_medium: 0.0727
map_large: -1.0000
mar_1: 0.0590
mar_10: 0.1629
mar_100: 0.2152
mar_small: 0.1636
mar_medium: 0.2356
mar_large: -1.0000
map_per_class: -1.0000
mar_100_per_class: -1.0000
classes: 0.0000

Model Metrics: Post layer 4 (Post-filter - VLM - Gemma3)
map: 0.0832
map_50: 0.2071
map_75: 0.0445
map_small: 0.0252
map_medium: 0.1088
map_large: -1.0000
mar_1: 0.0626
mar_10: 0.2052
mar_100: 0.2658
mar_small: 0.2977
mar_medium: 0.2532
mar_large: -1.0000
map_per_class: -1.0000
mar_100_per_class: -1.0000
classes: 0.0000

Model Metrics: Post layer 5 (SAM2 video tracking)
map: 0.0868
map_50: 0.2117
map_75: 0.0485
map_small: 0.0524
map_medium: 0.1020
map_large: -1.0000
mar_1: 0.0677
mar_10: 0.2274
mar_100: 0.2900
mar_small: 0.3216
mar_medium: 0.2775
mar_large: -1.0000
map_per_class: -1.0000
mar_100_per_class: -1.0000
classes: 0.0000

Model Metrics: Post layer 5 (SAM2 vide

# The End

In [27]:
"""


#Visualize the segment results across the video and save them

rows_final = []
box_labels = df["box_label"]
object_ids_v = []

if not os.path.exists(save_tracking_results_dir):
    os.makedirs(save_tracking_results_dir)

ID_TO_OBJECTS = {i: obj for i, obj in enumerate(box_labels, start=0)}

for frame_idx, segments in video_segments.items():
    img = cv2.imread(os.path.join(source_video_frame_dir, frame_names[frame_idx]))
    
    object_ids_v = list(segments.keys())
    masks = list(segments.values())
    masks = np.concatenate(masks, axis=0)
    
    detections = sv.Detections(
        xyxy=sv.mask_to_xyxy(masks),  # (n, 4)
        mask=masks, # (n, h, w)
        class_id=np.array(object_ids_v, dtype=np.int32),
    )
    box_annotator = sv.BoxAnnotator()
    annotated_frame = box_annotator.annotate(scene=img.copy(), detections=detections)
    label_annotator = sv.LabelAnnotator()
    annotated_frame = label_annotator.annotate(annotated_frame, detections=detections, labels=[str(ID_TO_OBJECTS[i]) for i in object_ids_v])
    #mask_annotator = sv.MaskAnnotator()
    #annotated_frame = mask_annotator.annotate(scene=annotated_frame, detections=detections)
    cv2.imwrite(os.path.join(save_tracking_results_dir, f"annotated_frame_{frame_idx:05d}.jpg"), annotated_frame)

    """


'\n\n\n#Visualize the segment results across the video and save them\n\nrows_final = []\nbox_labels = df["box_label"]\nobject_ids_v = []\n\nif not os.path.exists(save_tracking_results_dir):\n    os.makedirs(save_tracking_results_dir)\n\nID_TO_OBJECTS = {i: obj for i, obj in enumerate(box_labels, start=0)}\n\nfor frame_idx, segments in video_segments.items():\n    img = cv2.imread(os.path.join(source_video_frame_dir, frame_names[frame_idx]))\n    \n    object_ids_v = list(segments.keys())\n    masks = list(segments.values())\n    masks = np.concatenate(masks, axis=0)\n    \n    detections = sv.Detections(\n        xyxy=sv.mask_to_xyxy(masks),  # (n, 4)\n        mask=masks, # (n, h, w)\n        class_id=np.array(object_ids_v, dtype=np.int32),\n    )\n    box_annotator = sv.BoxAnnotator()\n    annotated_frame = box_annotator.annotate(scene=img.copy(), detections=detections)\n    label_annotator = sv.LabelAnnotator()\n    annotated_frame = label_annotator.annotate(annotated_frame, detectio

In [28]:
"""
#Step 6: Convert the annotated frames to video
 

create_video_from_images(save_tracking_results_dir, output_video_path)
"""

'\n#Step 6: Convert the annotated frames to video\n \n\ncreate_video_from_images(save_tracking_results_dir, output_video_path)\n'

In [29]:
"""
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import os

# Your folders
img_folder = source_video_frame_dir
pred_csv = model_preds_csv_5Layer
gt_csv = ground_truth_csv

# Load CSVs
pred_df = pd.read_csv(pred_csv, sep = ';')
gt_df = pd.read_csv(gt_csv, sep = ';')

# Function to draw boxes
def draw_boxes(img, boxes, color, label):
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

# Visualize few samples
for image_name in gt_df['image_name'].unique()[:5]:  # only first 5 images
    img_path = os.path.join(img_folder, image_name)
    image = cv2.imread(img_path)

    gt_boxes = gt_df[gt_df['image_name'] == image_name][["x_min", "y_min", "x_max", "y_max"]].values
    pred_boxes = pred_df[pred_df['image_name'] == image_name][["x_min", "y_min", "x_max", "y_max"]].values

    draw_boxes(image, gt_boxes, (0, 255, 0), "GT")      # Green for ground truth
    draw_boxes(image, pred_boxes, (255, 0, 0), "Pred")  # Blue for predictions

    # Convert BGR to RGB for matplotlib
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 8))
    plt.imshow(image_rgb)
    plt.title(f"Image: {image_name}")
    #plt.axis("off")
    plt.show()
"""
 


'\nimport cv2\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport os\n\n# Your folders\nimg_folder = source_video_frame_dir\npred_csv = model_preds_csv_5Layer\ngt_csv = ground_truth_csv\n\n# Load CSVs\npred_df = pd.read_csv(pred_csv, sep = \';\')\ngt_df = pd.read_csv(gt_csv, sep = \';\')\n\n# Function to draw boxes\ndef draw_boxes(img, boxes, color, label):\n    for box in boxes:\n        x1, y1, x2, y2 = map(int, box)\n        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)\n        cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)\n\n# Visualize few samples\nfor image_name in gt_df[\'image_name\'].unique()[:5]:  # only first 5 images\n    img_path = os.path.join(img_folder, image_name)\n    image = cv2.imread(img_path)\n\n    gt_boxes = gt_df[gt_df[\'image_name\'] == image_name][["x_min", "y_min", "x_max", "y_max"]].values\n    pred_boxes = pred_df[pred_df[\'image_name\'] == image_name][["x_min", "y_min", "x_max", "y_max"]].values\n\n    draw_bo

In [30]:

"""
# Optional: check unmatched image IDs if available
pred_image_ids = set([p["image_name"] for p in predictions])  # if applicable
gt_image_ids = set([g["image_name"] for g in ground_truth])
print("Unmatched IDs:", gt_image_ids - pred_image_ids)


print(len(predictions))
print(len(ground_truth))

output_path_csv = os.path.join(save_tracking_results_dir, "predictions.csv")
predictions_df = pd.DataFrame(predictions)
predictions_df.to_csv(output_path_csv, index=False, sep=";")

"""

'\n# Optional: check unmatched image IDs if available\npred_image_ids = set([p["image_name"] for p in predictions])  # if applicable\ngt_image_ids = set([g["image_name"] for g in ground_truth])\nprint("Unmatched IDs:", gt_image_ids - pred_image_ids)\n\n\nprint(len(predictions))\nprint(len(ground_truth))\n\noutput_path_csv = os.path.join(save_tracking_results_dir, "predictions.csv")\npredictions_df = pd.DataFrame(predictions)\npredictions_df.to_csv(output_path_csv, index=False, sep=";")\n\n'